# P1 — source-scoped pending review and conservative passage recovery

This isolated B v8 notebook replays verified cached measurements. It preserves B v7 duplicate/crossing controls and keeps the retained **machine baseline** separate from the **review-assisted** result. It does not run YOLO, tracking, ReID or enhancement. Results are research outputs; neither synthetic checks nor reviewed counts establish production accuracy.

Read `docs/B_V7_HANDOFF_CONTRACT.md` and the current P1 contract before execution. Original video files stay outside GitHub.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
HELPERS = REPO_ROOT / "notebooks"
if str(HELPERS) not in sys.path:
    sys.path.insert(0, str(HELPERS))
import phase_8_b_v8_batch as batch

Set the actual source directories. `SOURCE_RUN_ROOT` is the retained Phase 8 donor run directory containing the original manifest and `weekday_siang/track_records.jsonl.gz`; it is not the P1 output directory. The original video must have the same hash as the B v7 input. A missing input blocks replay rather than substituting a video. The baseline below is read from retained B v7 bytes, not recomputed.

In [ ]:
SOURCE_RUN_ROOT = None  # Set the actual mounted/downloaded verified donor run directory.
SOURCE_VIDEO_ROOT = Path("/kaggle/input/datasets/chrisbiran/tgdi-traffic-videos")
OVERLAY_PATH = REPO_ROOT / "results/b_v8/config/review_overlay_v5.json"
ADJUDICATIONS_PATH = REPO_ROOT / batch.ADJUDICATIONS_PATH
POLICY_PATH = REPO_ROOT / "results/b_v8/config/policy_v5.json"
RUN_REPLAY = False
RUN_ID = None  # None generates a new timestamped directory; existing runs are never overwritten.

In [ ]:
baseline, baseline_path, baseline_sha256 = batch.load_machine_baseline(REPO_ROOT)
print({"branch": "machine", "source": str(baseline_path), "sha256": baseline_sha256,
       "aggregate": baseline["aggregate"], "replayed_in_P1": False})

Preparation binds the exact B v7 policy, source/video/cache/geometry, review overlay and pending adjudications. Keep the reviewed claim origin explicit: user decisions and AI-assisted pixel review remain review inputs, not automatic model predictions. Config changes require a new isolated experiment rather than overwriting a frozen policy.

In [ ]:
expected_policy = batch.build_policy(REPO_ROOT, overlay_path=OVERLAY_PATH,
    adjudications_path=ADJUDICATIONS_PATH)
if POLICY_PATH.exists():
    assert batch._read(POLICY_PATH) == expected_policy, "Existing P1 policy differs; preserve it and create a separate experiment."
else:
    batch._write(POLICY_PATH, expected_policy)

assert SOURCE_RUN_ROOT is not None, "Set SOURCE_RUN_ROOT to the actual frozen donor run directory."
prepared = batch.prepare_b_v8_batch(repo_root=REPO_ROOT, source_run_root=SOURCE_RUN_ROOT,
    source_video_root=SOURCE_VIDEO_ROOT, overlay_path=OVERLAY_PATH,
    adjudications_path=ADJUDICATIONS_PATH, policy_path=POLICY_PATH)
print({"status": "prepared", "branch": prepared["branch"],
       "baseline_sha256": prepared["baseline_result_sha256"],
       "guarded_inputs": len(prepared["guarded_inputs"]),
       "new_model_execution": False})

Execute the complete 9,128-frame **review-assisted** replay only after preparation and the relevant regression tests pass. The original machine aggregate is a comparator; never add it to the reviewed aggregate. `decision_frame` retains candidate timing while provenance records actual offline EOF finalization. Review conflicts do not create measured crossing points during gaps.

In [ ]:
if RUN_REPLAY:
    result = batch.run_b_v8_batch(repo_root=REPO_ROOT, source_run_root=SOURCE_RUN_ROOT,
        source_video_root=SOURCE_VIDEO_ROOT, overlay_path=OVERLAY_PATH,
        adjudications_path=ADJUDICATIONS_PATH, policy_path=POLICY_PATH, run_id=RUN_ID)
    print(result["status"], result["output_directory"])
else:
    print("Replay not executed. Set RUN_REPLAY=True only after reviewing the frozen inputs and tests.")

Inspect the new run directory under `results/b_v8/execution/`: `batch_report.json`, `baseline_reference.json`, `weekday_siang/counting_result.json.gz`, source snapshots, review application audit, pending finalization audit, decision provenance and retention manifest. Reconcile every changed candidate with the B v7 source witnesses and the P1 triage inventory. Accepted, pending and invalid are different software states, not quantities to sum into physical traffic. Night-video generalization and model configuration changes are outside this replay.